In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Constants

In [2]:
# function name
str_function_name = 'genxii-ad-concat-tuning-2'

### 1. Create container

### Create ```Dockerfile```

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


### Write ```requirements.txt```

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0
pandas==1.2.4
boto3==1.24.59

Writing requirements.txt


### Write ```lambda_function.py```

In [5]:
%%writefile lambda_function.py

import pandas as pd
import boto3

# lambda handler
def lambda_handler(event, context):
    # constants
    str_project = '20231010-gen-xii'
    str_model = '01_ad'
    str_prefix = f'{str_model}/02_model/02_model/02_batch_tuning/models'
    
    # get df_hyperparameters
    str_filename = 'df_hyperparameters.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/12_step_function/{str_filename}'
    df = pd.read_csv(str_uri)
    # convert to dict
    dict_hyperparameters = dict(zip(df['keys'], df['values']))
    
    # get eval metric
    str_eval_metric = dict_hyperparameters['STR_EVAL_METRIC']
    print(f'Eval metric: {str_eval_metric}')
            
    # get index files in s3
    print('Getting files in s3...')
    cls_client = boto3.resource('s3')
    cls_bucket = cls_client.Bucket(str_project)
    list_str_filenames = []
    for file in cls_bucket.objects.filter(Prefix=str_prefix):
        # get key
        str_key = file.key
        # make sure it is a .csv
        if '.csv' in str_key:
            # get filename
            str_filename = str_key.split('/')[-1]
            list_str_filenames.append(str_filename)
    print(f'There are {len(list_str_filenames)} files to import')
    
    # iterate and import
    print('Importing files...')
    list_df = []
    for str_filename in list_str_filenames:
        str_uri = f's3://{str_project}/{str_prefix}/{str_filename}'
        df = pd.read_csv(str_uri)
        list_df.append(df)
    
    # create df
    print('Creating data frame...')
    df = pd.concat(list_df)
    del list_df
    
    # logic for sorting
    print('Sorting...')
    if str_eval_metric in ['AUC', 'PRAUC', 'F1']:
        bool_ascending = False
    else:
        bool_ascending = True
    df.sort_values(by='flt_eval_metric_valid', ascending=bool_ascending, inplace=True)
    
    # write to s3
    print('Writing to s3...')
    str_filename = 'df_tuning.csv'
    str_uri = f's3://{str_project}/{str_model}/02_model/02_model/03_lambda_concat_tuning/{str_filename}'
    df.to_csv(str_uri, index=False)

Writing lambda_function.py


### Build image and push to ECR

In [6]:
%%sh

# name the image
image=genxii-ad-concat-tuning-2

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  26.11kB
Step 1/6 : FROM public.ecr.aws/lambda/python:3.8
 ---> 3cd81ffec4d9
Step 2/6 : RUN pip install --upgrade pip
 ---> Using cache
 ---> 1fa305cd0a48
Step 3/6 : COPY requirements.txt  .
 ---> 8fcec98a1f24
Step 4/6 : RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"
 ---> Running in 62a87425e7dd
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.8/138.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.5/132.5 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 88.7 MB/s eta 0:00:00
   ━━━━━━━

Removing intermediate container 62a87425e7dd
 ---> 465f6f861306
Step 5/6 : COPY lambda_function.py ${LAMBDA_TASK_ROOT}
 ---> eb289764e0aa
Step 6/6 : CMD ["lambda_function.lambda_handler"]
 ---> Running in 67e871de6c39
Removing intermediate container 67e871de6c39
 ---> e38ef6f90272
Successfully built e38ef6f90272
Successfully tagged genxii-ad-concat-tuning-2:latest


WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'genxii-ad-concat-tuning-2' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-concat-tuning-2]
07d04ea624cd: Preparing
448cb05b759d: Preparing
35eeb59bf875: Preparing
3f97a2d36016: Preparing
e92756f7b561: Preparing
4fe51bf0bf5c: Preparing
fbbd8c1e2ec1: Preparing
fe2359fe88f2: Preparing
e703f2e518cc: Preparing
97a787951169: Preparing
4fe51bf0bf5c: Waiting
fbbd8c1e2ec1: Waiting
e703f2e518cc: Waiting
fe2359fe88f2: Waiting
97a787951169: Waiting
35eeb59bf875: Pushed
07d04ea624cd: Pushed
fbbd8c1e2ec1: Pushed
fe2359fe88f2: Pushed
3f97a2d36016: Pushed
e703f2e518cc: Pushed
e92756f7b561: Pushed
4fe51bf0bf5c: Pushed
97a787951169: Pushed
448cb05b759d: Pushed
latest: digest: sha256:37f8b7ed6ec64c93cf8f546456bb83eafe31d8a5b05fe01d96768cfc3d510155 size: 2419


### 2. Create lambda function from image

In [7]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [8]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [9]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 22 Apr 2024 19:40:24 GMT',
                                      'x-amzn-requestid': 'f73e4a69-d05a-49d0-9257-5cea34f435a6'},
                      'HTTPStatusCode': 204,
                      'RequestId': 'f73e4a69-d05a-49d0-9257-5cea34f435a6',
                      'RetryAttempts': 0}}


In [10]:
# create function
str_image_uri = '836690756591.dkr.ecr.us-west-2.amazonaws.com/genxii-ad-concat-tuning-2:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=60,
    MemorySize=512,
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 512,
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '37f8b7ed6ec64c93cf8f546456bb83eafe31d8a5b05fe01d96768cfc3d510155',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 512},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:genxii-ad-concat-tuning-2',
 'FunctionName': 'genxii-ad-concat-tuning-2',
 'LastModified': '2024-04-22T19:40:24.496+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/genxii-ad-concat-tuning-2'},
 'MemorySize': 512,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1207',
                                      'content-type': 'application/json',
                                      'date': 'Mon, 22 Apr 2024 19:40:25 GMT',
                                      'x-amzn-requestid': 'b6f59da0-648b-485f-b63b-eb52b1f1561e'},
                      'HTTPStatusCode': 201,
                      'RequestId': 'b6f59da

### Clean-up

In [11]:
for str_file in ['Dockerfile', 'lambda_function.py', 'requirements.txt']:
    os.remove(str_file)